In [6]:
import os
import sys
import ctypes
from pathlib import Path
import numpy as np
import pandas as pd
import napari
import tifffile as tif
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Make NVRTC/NVIDIA DLLs discoverable before importing CuPy.
if os.name == 'nt':
    torch_lib = Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib'
    dll_dirs = [
        Path(sys.prefix) / 'Library' / 'bin',
        torch_lib,
    ]
    for dll_dir in dll_dirs:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

    if torch_lib.exists():
        os.environ['PATH'] = f"{torch_lib};" + os.environ.get('PATH', '')
        for name in ('nvrtc64_120_0.dll', 'nvrtc-builtins64_128.dll'):
            dll_path = torch_lib / name
            if dll_path.exists():
                ctypes.WinDLL(str(dll_path))

import cupy as cp
import cupyx.scipy.ndimage as ndi_gpu
import dask.array as da
from skimage.morphology import skeletonize as skeletonize_3d
import sknw
from scipy.spatial.distance import cdist
from scipy.ndimage import distance_transform_edt as edt
from scipy import ndimage as ndi

In [7]:
def add_grouped_stats(df, group_masks, combined_name, exclude_cols):
    stats_out = {}
    for key, value in df.items():
        if key in exclude_cols:
            continue
        if not pd.api.types.is_numeric_dtype(value):
            continue

        arr = pd.to_numeric(value, errors='coerce').to_numpy(dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size == 0:
            combined_mean, combined_std, combined_median = np.nan, np.nan, np.nan
        else:
            combined_mean, combined_std, combined_median = float(np.mean(arr)), float(np.std(arr)), float(np.median(arr))

        stats_out[f'mean_{combined_name}_{key}'] = combined_mean
        stats_out[f'std_{combined_name}_{key}'] = combined_std
        stats_out[f'median_{combined_name}_{key}'] = combined_median

        for group_name, group_mask in group_masks.items():
            g_arr = pd.to_numeric(value[group_mask], errors='coerce').to_numpy(dtype=float)
            g_arr = g_arr[~np.isnan(g_arr)]
            if g_arr.size == 0:
                group_mean, group_std, group_median = np.nan, np.nan, np.nan
            else:
                group_mean, group_std, group_median = float(np.mean(g_arr)), float(np.std(g_arr)), float(np.median(g_arr))

            stats_out[f'mean_{group_name}_{key}'] = group_mean
            stats_out[f'std_{group_name}_{key}'] = group_std
            stats_out[f'median_{group_name}_{key}'] = group_median

    return stats_out

In [8]:
def cupy_chunk_processing(volume, processing_func, chunk_size=(64, 512, 512), overlap=(15, 15, 15), *args, **kwargs):
    result = np.empty_like(volume)
    pool = cp.get_default_memory_pool()
    z_steps = range(0, volume.shape[0], chunk_size[0])

    for z in tqdm(z_steps, desc='Processing chunks'):
        for y in range(0, volume.shape[1], chunk_size[1]):
            for x in range(0, volume.shape[2], chunk_size[2]):
                z_start = max(0, z - overlap[0])
                z_end = min(volume.shape[0], z + chunk_size[0] + overlap[0])
                y_start = max(0, y - overlap[1])
                y_end = min(volume.shape[1], y + chunk_size[1] + overlap[1])
                x_start = max(0, x - overlap[2])
                x_end = min(volume.shape[2], x + chunk_size[2] + overlap[2])

                chunk = volume[z_start:z_end, y_start:y_end, x_start:x_end]
                chunk_gpu = cp.asarray(chunk)

                def func_wrapper(gpu_chunk):
                    gpu_args = []
                    for arg in args:
                        gpu_args.append(cp.asarray(arg) if isinstance(arg, np.ndarray) else arg)

                    gpu_kwargs = {}
                    for k, v in kwargs.items():
                        gpu_kwargs[k] = cp.asarray(v) if isinstance(v, np.ndarray) else v

                    return processing_func(gpu_chunk, *gpu_args, **gpu_kwargs)

                filtered_chunk = func_wrapper(chunk_gpu)

                w_z_start = z - z_start
                w_z_end = w_z_start + chunk_size[0]
                w_y_start = y - y_start
                w_y_end = w_y_start + chunk_size[1]
                w_x_start = x - x_start
                w_x_end = w_x_start + chunk_size[2]

                valid_chunk = filtered_chunk[
                    w_z_start:min(w_z_end, filtered_chunk.shape[0]),
                    w_y_start:min(w_y_end, filtered_chunk.shape[1]),
                    w_x_start:min(w_x_end, filtered_chunk.shape[2]),
                ].get()

                result_z = min(z, result.shape[0])
                result_y = min(y, result.shape[1])
                result_x = min(x, result.shape[2])

                result[
                    result_z:result_z + valid_chunk.shape[0],
                    result_y:result_y + valid_chunk.shape[1],
                    result_x:result_x + valid_chunk.shape[2],
                ] = valid_chunk

                del chunk_gpu, filtered_chunk
                pool.free_all_blocks()

    return result

def measure_edge_length(coordinates):
    differences = np.diff(coordinates, axis=0)
    segment_lengths = np.linalg.norm(differences, axis=1)
    return np.sum(segment_lengths)

def prune_graph(graph, area_3d, edt_cutoff=0.25, length_cutoff=25):
    while True:
        endpoint_nodes = [node for node, degree in graph.degree() if degree == 1]
        values = []
        for node in endpoint_nodes:
            neighbors = list(graph.neighbors(node))

            if len(neighbors) == 1:
                neighbor = neighbors[0]
                edge_data = graph.get_edge_data(neighbor, node)
                edge_length = measure_edge_length(edge_data['pts'])
                branch_edt = area_3d[edge_data['pts'][:, 0], edge_data['pts'][:, 1], edge_data['pts'][:, 2]]
                branch_edt_interp = np.interp(np.linspace(0, 1, 100), np.linspace(0, 1, branch_edt.size), branch_edt)

                if np.mean(branch_edt_interp[:50]) < np.mean(branch_edt_interp[-50:]):
                    neighbor_coords = edge_data['pts'][-1]
                    part_oi = branch_edt_interp[:20]
                else:
                    neighbor_coords = edge_data['pts'][0]
                    part_oi = branch_edt_interp[-20:]

                neighbor_edt = area_3d[neighbor_coords[0], neighbor_coords[1], neighbor_coords[2]]
                value = np.mean(part_oi) / (neighbor_edt + 1e-6)

                if value > edt_cutoff or edge_length <= length_cutoff:
                    graph.remove_node(node)
                    values.append(value)

        if len(values) == 0:
            break
    return graph

def remove_mid_node(graph):
    while True:
        nodes_to_process = [n for n, d in graph.degree() if d == 2]
        if not nodes_to_process:
            break

        processed_in_iteration = False
        for i in nodes_to_process:
            if not graph.has_node(i) or graph.degree(i) != 2:
                continue

            neighbors = list(graph.neighbors(i))
            if len(neighbors) != 2:
                continue

            n1, n2 = neighbors[0], neighbors[1]
            if n1 == n2 or graph.has_edge(n1, n2):
                continue

            edge1 = graph.get_edge_data(i, n1)
            edge2 = graph.get_edge_data(i, n2)
            pts1 = np.atleast_2d(edge1['pts'])
            pts2 = np.atleast_2d(edge2['pts'])
            node_coord = graph.nodes[i]['pts'].astype(np.int32)

            s1, e1 = pts1[0], pts1[-1]
            s2, e2 = pts2[0], pts2[-1]

            dists = cdist([s1, e1], [s2, e2])
            min_row, min_col = np.unravel_index(np.argmin(dists), dists.shape)

            if min_row == 0 and min_col == 0:
                combined_line = np.concatenate([pts1[::-1], [node_coord], pts2], axis=0)
            elif min_row == 1 and min_col == 1:
                combined_line = np.concatenate([pts1, [node_coord], pts2[::-1]], axis=0)
            elif min_row == 0 and min_col == 1:
                combined_line = np.concatenate([pts2[::-1], [node_coord], pts1], axis=0)
            else:
                combined_line = np.concatenate([pts1, [node_coord], pts2], axis=0)

            new_weight = edge1.get('weight', 0) + edge2.get('weight', 0)
            graph.add_edge(n1, n2, weight=new_weight, pts=combined_line)
            graph.remove_node(i)
            processed_in_iteration = True

        if not processed_in_iteration:
            break
    return graph

def collect_border_vicinity_edges(graph, image_shape, vicinity_z=1, vicinity_xy=50):
    border_vicinity_edges = set()
    for u, v in graph.edges():
        try:
            pts = graph[u][v]['pts']
            if any((
                # pt[0] < vicinity_z or pt[0] > image_shape[0] - 1 - vicinity_z or
                pt[1] < vicinity_xy or pt[1] > image_shape[1] - 1 - vicinity_xy or
                pt[2] < vicinity_xy or pt[2] > image_shape[2] - 1 - vicinity_xy
                ) for pt in pts):
                border_vicinity_edges.add((u, v))
        except KeyError:
            continue

    trimmed_subgraph = graph.copy()
    edges_to_remove = [edge for edge in border_vicinity_edges if trimmed_subgraph.has_edge(*edge)]
    trimmed_subgraph.remove_edges_from(edges_to_remove)

    isolated_nodes = [node for node in trimmed_subgraph.nodes() if trimmed_subgraph.degree[node] == 0]
    if isolated_nodes:
        trimmed_subgraph.remove_nodes_from(isolated_nodes)

    return trimmed_subgraph

def compute_cross_sectional_areas(mask, skeleton, binary_edt):
    edt_2d = edt(np.max(mask, axis=0))
    area_3d = np.zeros_like(binary_edt, dtype=float)
    z_idx, y_idx, x_idx = np.where(skeleton > 0)

    minor_axis = 2 * binary_edt[z_idx, y_idx, x_idx]
    major_axis = 2 * edt_2d[y_idx, x_idx]

    areas = np.pi * (major_axis / 2) * (minor_axis / 2)
    area_3d[z_idx, y_idx, x_idx] = areas
    return area_3d

def compute_vessel_metrics(graph, area_image, vessel_metrics_df):
    zs, ys, xs = [], [], []
    volumes, lengths, shortest_paths = [], [], []
    tortuosities, is_sprouts = [], []
    mean_cs_areas, median_cs_areas, std_cs_areas = [], [], []
    node1_degrees, node2_degrees = [], []
    orientation_zs, orientation_ys, orientation_xs = [], [], []

    edges = list(graph.edges())
    valid_edges = []
    if not edges:
        cols = [
            'z', 'y', 'x', 'volume', 'length', 'shortest_path', 'tortuosity',
            'is_sprout', 'mean_cs_area', 'median_cs_area', 'std_cs_area',
            'node1_degree', 'node2_degree', 'orientation_z', 'orientation_y', 'orientation_x'
        ]
        return pd.DataFrame(columns=cols)

    node_degrees_dict = dict(graph.degree())

    for u, v in edges:
        try:
            pts = graph[u][v]['pts']
            if len(pts) < 2:
                continue

            valid_edges.append((u, v))
            zs.append(pts[:, 0])
            ys.append(pts[:, 1])
            xs.append(pts[:, 2])

            segment_areas = area_image[pts[:, 0], pts[:, 1], pts[:, 2]]
            mean_cs_areas.append(np.nanmean(segment_areas))
            median_cs_areas.append(np.nanmedian(segment_areas))
            std_cs_areas.append(np.nanstd(segment_areas))
            volumes.append(np.nansum(segment_areas))

            l = measure_edge_length(pts)
            lengths.append(l)
            sp = np.linalg.norm(pts[0] - pts[-1])
            shortest_paths.append(sp)
            tortuosities.append(np.clip(l / (sp + 1e-8), 0, 5))

            deg_u = node_degrees_dict.get(u, 0)
            deg_v = node_degrees_dict.get(v, 0)
            node1_degrees.append(deg_u)
            node2_degrees.append(deg_v)

            is_sprouts.append(graph.nodes[u]['sprout'] or graph.nodes[v]['sprout'])

            direction_vector = pts[-1] - pts[0]
            norm = np.linalg.norm(direction_vector)
            if norm > 1e-8:
                normalized_vector = direction_vector / norm
            else:
                normalized_vector = np.array([np.nan, np.nan, np.nan])

            orientation_zs.append(normalized_vector[0])
            orientation_ys.append(normalized_vector[1])
            orientation_xs.append(normalized_vector[2])
        except (KeyError, IndexError):
            continue

    vessel_metrics_df = pd.DataFrame(index=pd.MultiIndex.from_tuples(valid_edges, names=['node1', 'node2']))

    vessel_metrics_df['z'] = zs
    vessel_metrics_df['y'] = ys
    vessel_metrics_df['x'] = xs
    vessel_metrics_df['volume'] = volumes
    vessel_metrics_df['length'] = lengths
    vessel_metrics_df['shortest_path'] = shortest_paths
    vessel_metrics_df['tortuosity'] = tortuosities
    vessel_metrics_df['is_sprout'] = is_sprouts
    vessel_metrics_df['mean_cs_area'] = mean_cs_areas
    vessel_metrics_df['median_cs_area'] = median_cs_areas
    vessel_metrics_df['std_cs_area'] = std_cs_areas
    vessel_metrics_df['node1_degree'] = node1_degrees
    vessel_metrics_df['node2_degree'] = node2_degrees
    vessel_metrics_df['orientation_z'] = orientation_zs
    vessel_metrics_df['orientation_y'] = orientation_ys
    vessel_metrics_df['orientation_x'] = orientation_xs

    return vessel_metrics_df

def compute_junction_metrics(graph, junction_metrics_df, distance_threshold=50):
    cols = ['z', 'y', 'x', 'number_of_vessel_per_node', 'node_type', 'dist_nearest_junction', 'dist_nearest_endpoint', 'num_junction_neighbors', 'num_endpoint_neighbors']
    nodes = list(graph.nodes())
    if not nodes:
        return pd.DataFrame(index=nodes, columns=cols)

    positions = np.array([graph.nodes[n]['pts'] for n in nodes])
    node_type = np.array(['sprout' if graph.nodes[n]['sprout'] else 'junction' for n in nodes])
    junction_metrics_df = pd.DataFrame(index=nodes)
    junction_metrics_df['z'] = positions[:, 0]
    junction_metrics_df['y'] = positions[:, 1]
    junction_metrics_df['x'] = positions[:, 2]
    junction_metrics_df['number_of_vessel_per_node'] = np.array([graph.degree[n] for n in nodes])
    junction_metrics_df['node_type'] = node_type  # endpoint ('sprout') or branching node ('junction')
    junction_metrics_df['dist_nearest_junction'] = np.nan  # nearest distance to a junction node
    junction_metrics_df['dist_nearest_endpoint'] = np.nan  # nearest distance to a sprout endpoint
    junction_metrics_df['num_junction_neighbors'] = 0  # number of junctions within threshold
    junction_metrics_df['num_endpoint_neighbors'] = 0  # number of endpoints within threshold

    if len(nodes) < 2:
        return junction_metrics_df

    dist_matrix = cdist(positions, positions)
    endpoint_mask = (node_type == 'sprout')
    branch_point_mask = (node_type == 'junction')

    def fill_group_metrics(target_mask, reference_mask, dist_col, count_col, same_group=False):
        if not np.any(target_mask) or not np.any(reference_mask):
            return
        sub_dist = dist_matrix[np.ix_(target_mask, reference_mask)].copy()
        if same_group:
            np.fill_diagonal(sub_dist, np.inf)
        junction_metrics_df.loc[target_mask, dist_col] = np.min(sub_dist, axis=1)
        junction_metrics_df.loc[target_mask, count_col] = np.sum(sub_dist <= distance_threshold, axis=1)

    fill_group_metrics(branch_point_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors', same_group=True)
    fill_group_metrics(branch_point_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors')
    fill_group_metrics(endpoint_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors')
    fill_group_metrics(endpoint_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors', same_group=True)
    return junction_metrics_df

def fractal_dimension_and_lacunarity(binary, min_box_size=1, max_box_size=None, n_samples=12):
    pts = np.argwhere(binary > 0)
    if pts.size == 0:
        return np.nan, np.nan

    if max_box_size is None:
        max_box_size = int(np.floor(np.log2(np.min(binary.shape))))

    scales = np.floor(np.logspace(max_box_size, min_box_size, num=n_samples, base=2)).astype(np.int64)
    scales = np.unique(scales)
    scales = scales[scales > 0]

    log_inv_scale = []
    log_N = []
    lac_vals = []

    for s in scales:
        box_ids = pts // s
        unique_box_ids, counts = np.unique(box_ids, axis=0, return_counts=True)  # occupied boxes + masses
        N = unique_box_ids.shape[0]
        if N < 2:
            continue

        log_inv_scale.append(np.log(1.0 / s))
        log_N.append(np.log(N))

        mu = counts.mean()
        lac_vals.append((counts.var() / (mu * mu)) if mu > 0 else np.nan)

    if len(log_N) < 2:
        return np.nan, np.nan

    fd = float(np.polyfit(log_inv_scale, log_N, 1)[0])
    lac = float(np.nanmean(lac_vals)) if len(lac_vals) else np.nan
    return fd, lac


def graph2image(graph, shape):
    pruned_skeleton = np.zeros(shape)
    for u, v in graph.edges():
        coords = graph.get_edge_data(u, v)['pts']

        clipped_coords = np.zeros_like(coords)
        clipped_coords[:, 0] = np.clip(coords[:, 0], 0, shape[0] - 1)
        clipped_coords[:, 1] = np.clip(coords[:, 1], 0, shape[1] - 1)
        clipped_coords[:, 2] = np.clip(coords[:, 2], 0, shape[2] - 1)

        pruned_skeleton[clipped_coords[:, 0], clipped_coords[:, 1], clipped_coords[:, 2]] = 1

    return pruned_skeleton

In [9]:
vasculature_segmentation= np.load(r"C:\Users\taylorhearn\git_repos\vascumap\working_outputs_double_crop\20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10_vessel_mask_iso_354_cropped.npy")

In [10]:
##################### CLEAN SEGMENTATION #####################
chunk_size = (vasculature_segmentation.shape[0], 512, 512)
binary_smoothed = cupy_chunk_processing(volume=vasculature_segmentation.astype(np.float32), processing_func=ndi_gpu.gaussian_filter,sigma=3, chunk_size=chunk_size) > 0.5
binary_filled_holes = cupy_chunk_processing(volume=binary_smoothed, processing_func=ndi_gpu.binary_fill_holes, chunk_size=chunk_size).astype(np.float32)
binary_edt = cupy_chunk_processing(volume=binary_filled_holes, processing_func=ndi_gpu.distance_transform_edt, chunk_size=chunk_size)

##################### SKELETONISE AND GRAPH #####################
dask_volume = da.from_array(binary_filled_holes.astype(bool), chunks=chunk_size)
skeleton = da.overlap.map_overlap(dask_volume, skeletonize_3d, depth=(2, 2, 2), boundary='none', dtype=bool).compute(scheduler='threads')
graph = sknw.build_sknw(skeleton)
area_image = compute_cross_sectional_areas(vasculature_segmentation, skeleton, binary_edt) # colour = area of vasculature at that point on skeleton

for n in list(graph.nodes()):
    graph.nodes[n]['pts'] = graph.nodes[n]['pts'][0]
    graph.nodes[n]['sprout'] = graph.degree(n) == 1

pruned_graph = prune_graph(graph=graph, area_3d=area_image, edt_cutoff=0.20, length_cutoff=25)
clean_graph = remove_mid_node(pruned_graph)
clean_graph = collect_border_vicinity_edges(clean_graph, vasculature_segmentation.shape)
isolated_nodes = [node for node in clean_graph.nodes() if clean_graph.degree[node] == 0]
if isolated_nodes:
    clean_graph.remove_nodes_from(isolated_nodes)

# Convert final cleaned graph back to a 3D skeleton volume
skeleton_from_graph = graph2image(clean_graph, vasculature_segmentation.shape).astype(np.uint8)
for node in clean_graph.nodes():
    clean_graph.nodes[node]['sprout'] = clean_graph.degree(node) == 1

##################### METRICS #####################
junction_metrics_df = pd.DataFrame()
vessel_metrics_df = pd.DataFrame()
global_metrics = {}

if clean_graph.number_of_edges() > 0:
    fd, lacunarity = fractal_dimension_and_lacunarity(area_image > 0)
    total_vessel_length = np.sum([np.linalg.norm(np.diff(clean_graph[u][v]['pts'], axis=0), axis=1).sum() for u, v in clean_graph.edges()])
    branchpoints_count = sum(1 for u in clean_graph.nodes() if not clean_graph.nodes[u]['sprout'])
    vessels_count = sum(1 for u, v in clean_graph.edges() if not clean_graph.nodes[u]['sprout'] and not clean_graph.nodes[v]['sprout'])
    sprouts_count = clean_graph.number_of_edges() - vessels_count
else:
    fd, lacunarity = np.nan, np.nan
    total_vessel_length = 0
    branchpoints_count = 0
    vessels_count = 0
    sprouts_count = 0

global_metrics['total_vessel_length'] = total_vessel_length  # sum of all centerline segment lengths
global_metrics['total_number_of_sprouts'] = sprouts_count  # edges connected to at least one endpoint
global_metrics['total_number_of_branches'] = vessels_count  # edges between non-endpoint nodes
global_metrics['total_number_of_junctions'] = branchpoints_count  # count of non-endpoint nodes
global_metrics['fractal_dimension'] = fd  # scaling complexity from box counting
global_metrics['lacunarity'] = lacunarity  # heterogeneity/gappiness of occupied boxes

if clean_graph.number_of_nodes() > 0 and clean_graph.number_of_edges() > 0:
    vessel_metrics_df = compute_vessel_metrics(clean_graph, area_image, vessel_metrics_df)
    junction_metrics_df = compute_junction_metrics(clean_graph, junction_metrics_df, distance_threshold=250)

    if not vessel_metrics_df.empty:
        is_sprout_mask = vessel_metrics_df['is_sprout']
        vessel_exclude = {
            'z', 'y', 'x', 'is_sprout', 'mean_cs_area', 'median_cs_area',
            'std_cs_area', 'node1_degree', 'node2_degree', 'orientation_z',
            'orientation_y', 'orientation_x', 'sample'
        }
        global_metrics.update(add_grouped_stats(vessel_metrics_df, group_masks={'branch': ~is_sprout_mask, 'sprout': is_sprout_mask}, combined_name='sprout_and_branch', exclude_cols=vessel_exclude))

    if not junction_metrics_df.empty:
        is_branch_point_mask = (junction_metrics_df['node_type'] == 'junction')
        is_sprout_tip_mask = (junction_metrics_df['node_type'] == 'sprout')
        junction_exclude = {'z', 'y', 'x', 'node_type', 'sample'}
        global_metrics.update(add_grouped_stats(junction_metrics_df, group_masks={'junction': is_branch_point_mask, 'sprout': is_sprout_tip_mask}, combined_name='junction_and_sprout', exclude_cols=junction_exclude))

global_metrics_df = pd.DataFrame([global_metrics])

Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\dask\array\overlap.py:667: FutureWarning: The use of map_overlap(array, func, **kwargs) is deprecated since dask 2.17.0 and will be an error in a future release. To silence this warning, use the syntax map_overlap(func, array0,[ array1, ...,] **kwargs) instead.
  warnings.warn(


In [22]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation")
viewer.add_labels(binary_filled_holes.astype(np.uint8), name="filled")
viewer.add_labels(skeleton.astype(np.uint8), name="raw_skeleton")
viewer.add_labels(skeleton_from_graph, name="pruned_graph_skeleton")

# Optional: show graph nodes as points
node_pts = np.array([clean_graph.nodes[n]["pts"] for n in clean_graph.nodes()], dtype=float)
if len(node_pts) > 0:
    viewer.add_points(node_pts, name="graph_nodes")

In [41]:
# Holes are background voxels (not bitwise inversion of integer labels)
holes = (vasculature_segmentation == 0)

# Compute EDT of holes with the same chunked GPU pipeline
hole_distance = cupy_chunk_processing(
    volume=holes.astype(np.float32),
    processing_func=ndi_gpu.distance_transform_edt,
    chunk_size=chunk_size
    )

Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.61s/it]


In [42]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation", opacity=0.2)
viewer.add_labels(holes.astype(np.uint8), name="holes")
viewer.add_image(hole_distance.astype(np.float32), name="hole_distance")

<Image layer 'hole_distance' at 0x18c7b0f3e50>

In [51]:
import numpy as np
import pandas as pd

from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max


def detect_pores_per_slice(mask,
                           voxel_size_xy=1.0,
                           voxel_size_z=1.0,
                           min_radius=2,
                           min_peak_distance=5):
    """
    Detect pores per slice using distance transform maxima.

    Parameters
    ----------
    mask : 3D ndarray
        Binary vessel mask (1=vessel, 0=background)

    Returns
    -------
    pores_df : pandas DataFrame
        Detected pore centres and radii
    """

    pores = []
    Z = mask.shape[0]

    for z in range(Z):
        vessel_slice = mask[z].astype(bool)
        background = ~vessel_slice

        dist = distance_transform_edt(background) * voxel_size_xy

        peaks = peak_local_max(
            dist,
            min_distance=min_peak_distance,
            threshold_abs=min_radius,
            exclude_border=True
        )

        for y, x in peaks:
            r = dist[y, x]
            pores.append({
                "z": z,
                "y": y,
                "x": x,
                "radius": r
            })

    pores_df = pd.DataFrame(pores)
    return pores_df


def compute_slice_statistics(pores_df):
    """
    Compute pore size distribution statistics per slice.
    """

    if pores_df.empty:
        return pd.DataFrame(columns=[
            "z", "n_pores", "mean_radius", "median_radius", "std_radius", "p90_radius", "max_radius"
        ])

    stats = []
    for z, g in pores_df.groupby("z"):
        r = g.radius.values
        stats.append({
            "z": z,
            "n_pores": len(r),
            "mean_radius": np.mean(r),
            "median_radius": np.median(r),
            "std_radius": np.std(r),
            "p90_radius": np.percentile(r, 90) if len(r) > 0 else np.nan,
            "max_radius": np.max(r) if len(r) > 0 else np.nan
        })

    return pd.DataFrame(stats)


def link_pores_to_cylinders(pores_df,
                            voxel_size_xy=1.0,
                            voxel_size_z=1.0,
                            max_xy_distance=16,
                            max_radius_change=8,
                            max_z_gap=3):
    """
    Link pores across slices into cylinders.

    Returns
    -------
    cylinders_df : DataFrame
    tracks : list[dict]
        Each track has {'id': int, 'points': list[pandas.Series]}
    """

    if pores_df.empty:
        return pd.DataFrame(), []

    pores_df = pores_df.sort_values("z").reset_index(drop=True)

    active_tracks = []
    current_id = 1

    for z in sorted(pores_df.z.unique()):
        slice_pores = pores_df[pores_df.z == z]

        if len(active_tracks) == 0:
            for _, p in slice_pores.iterrows():
                active_tracks.append({
                    "id": current_id,
                    "points": [p],
                })
                current_id += 1
            continue

        used = set()

        for track in active_tracks:
            last = track["points"][-1]
            candidates = slice_pores.copy()
            if len(candidates) == 0:
                continue

            dxy = np.sqrt(
                (candidates.x - last.x) ** 2 +
                (candidates.y - last.y) ** 2
            )
            dz = np.abs(candidates.z - last.z)
            dr = np.abs(candidates.radius - last.radius)

            valid = candidates[
                (dxy <= max_xy_distance) &
                (dz <= max_z_gap) &
                (dr <= max_radius_change)
            ]

            if len(valid) == 0:
                continue

            best_idx = dxy[valid.index].idxmin()
            best = valid.loc[best_idx]

            track["points"].append(best)
            used.add(best_idx)

        for idx, p in slice_pores.iterrows():
            if idx not in used:
                active_tracks.append({
                    "id": current_id,
                    "points": [p],
                })
                current_id += 1

    cyl_records = []
    for track in active_tracks:
        pts = track["points"]
        if len(pts) < 2:
            continue

        z_vals = [p.z for p in pts]
        r_vals = [p.radius for p in pts]

        length = (max(z_vals) - min(z_vals) + 1) * voxel_size_z
        volume = np.sum([np.pi * (r ** 2) * voxel_size_z for r in r_vals])

        cyl_records.append({
            "cylinder_id": track["id"],
            "z_start": int(min(z_vals)),
            "z_end": int(max(z_vals)),
            "length_um": float(length),
            "mean_radius": float(np.mean(r_vals)),
            "radius_std": float(np.std(r_vals)),
            "volume_est": float(volume),
            "n_points": int(len(pts)),
        })

    cylinders_df = pd.DataFrame(cyl_records)
    return cylinders_df, active_tracks


def _paint_disk_2d(image_2d, cy, cx, radius, label_value):
    h, w = image_2d.shape
    y_min = max(0, int(np.floor(cy - radius)))
    y_max = min(h - 1, int(np.ceil(cy + radius)))
    x_min = max(0, int(np.floor(cx - radius)))
    x_max = min(w - 1, int(np.ceil(cx + radius)))

    yy, xx = np.ogrid[y_min:y_max + 1, x_min:x_max + 1]
    mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2
    image_2d[y_min:y_max + 1, x_min:x_max + 1][mask] = label_value


def build_cylinder_label_image(mask_shape, tracks, radius_scale=1.25):
    """
    Create a 3D label image where each linked cylinder has a unique integer ID.
    Paints detections and interpolates between consecutive slices for continuity.
    """
    labels = np.zeros(mask_shape, dtype=np.int32)

    for track in tracks:
        cid = int(track["id"])
        pts = track["points"]
        if len(pts) < 2:
            continue

        for p in pts:
            z = int(round(p.z))
            y = float(p.y)
            x = float(p.x)
            r = max(1.0, float(p.radius) * float(radius_scale))

            if 0 <= z < labels.shape[0]:
                _paint_disk_2d(labels[z], y, x, r, cid)

        for p0, p1 in zip(pts[:-1], pts[1:]):
            z0, y0, x0, r0 = float(p0.z), float(p0.y), float(p0.x), float(p0.radius)
            z1, y1, x1, r1 = float(p1.z), float(p1.y), float(p1.x), float(p1.radius)

            n_steps = max(1, int(abs(round(z1) - round(z0))))
            for t in np.linspace(0.0, 1.0, n_steps + 1):
                z = int(round((1 - t) * z0 + t * z1))
                y = (1 - t) * y0 + t * y1
                x = (1 - t) * x0 + t * x1
                r = max(1.0, ((1 - t) * r0 + t * r1) * float(radius_scale))
                if 0 <= z < labels.shape[0]:
                    _paint_disk_2d(labels[z], y, x, r, cid)

    return labels


def analyse_vascular_pores(mask,
                           voxel_size_xy=1.0,
                           voxel_size_z=1.0,
                           min_radius=2,
                           min_peak_distance=5,
                           max_xy_distance=16,
                           max_radius_change=8,
                           max_z_gap=3,
                           label_radius_scale=1.25):

    pores_df = detect_pores_per_slice(
        mask,
        voxel_size_xy,
        voxel_size_z,
        min_radius,
        min_peak_distance
    )

    slice_stats = compute_slice_statistics(pores_df)

    cylinders_df, tracks = link_pores_to_cylinders(
        pores_df,
        voxel_size_xy,
        voxel_size_z,
        max_xy_distance=max_xy_distance,
        max_radius_change=max_radius_change,
        max_z_gap=max_z_gap,
    )

    cylinders_label_image = build_cylinder_label_image(
        mask.shape,
        tracks,
        radius_scale=label_radius_scale,
    )

    return pores_df, slice_stats, cylinders_df, cylinders_label_image


# ----------------------------
# Example usage
# ----------------------------

pores, slice_stats, cylinders, cylinders_label_image = analyse_vascular_pores(
    vasculature_segmentation,
    voxel_size_xy=1.0,
    voxel_size_z=2.0,
    min_radius=2,
    min_peak_distance=5,
    max_xy_distance=16,
    max_radius_change=8,
    max_z_gap=3,
    label_radius_scale=1.25,
)

print("Detected pores:", len(pores))
print("Detected cylinders:", len(cylinders))
print("Cylinder label IDs:", int(cylinders_label_image.max()))

print(slice_stats.head())

Detected pores: 84692
Detected cylinders: 11430
Cylinder label IDs: 13105
   z  n_pores  mean_radius  median_radius  std_radius  p90_radius  max_radius
0  0     1897    15.038276      11.661904   12.933591   30.000000   95.015788
1  1     1906    14.719945      11.180340   12.796824   29.154759   95.524866
2  2     1833    14.933992      12.000000   12.727682   29.342483   95.257546
3  3     1816    14.968158      12.000000   12.558824   30.000000   95.257546
4  4     1774    15.238516      12.083046   12.497086   30.000000   92.439169


In [52]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation", opacity=0.2)
viewer.add_labels(holes.astype(np.uint8), name="holes", opacity=0.1)
viewer.add_image(hole_distance.astype(np.float32), name="hole_distance", opacity=0.35, blending="additive")

if 'cylinders_label_image' in globals():
    viewer.add_labels(
        cylinders_label_image.astype(np.int32),
        name="cylinders_label_image",
        opacity=0.55,
    )

if len(pores) > 0:
    pore_points = pores[["z", "y", "x"]].to_numpy(dtype=float)
    pore_sizes = np.clip(2.0 * pores["radius"].to_numpy(dtype=float), 1.0, 20.0)
    points_layer = viewer.add_points(
        pore_points,
        name="pores",
        size=pore_sizes,
        opacity=0.9,
    )
    if hasattr(points_layer, "face_color"):
        points_layer.face_color = "cyan"
    if hasattr(points_layer, "border_color"):
        points_layer.border_color = "white"
    elif hasattr(points_layer, "edge_color"):
        points_layer.edge_color = "white"

print(
    f"Overlay loaded: {len(pores)} pores, {len(cylinders)} cylinders, "
    f"{int(cylinders_label_image.max()) if 'cylinders_label_image' in globals() else 0} label IDs."
)

Overlay loaded: 84692 pores, 11430 cylinders, 13105 label IDs.
